# Agentic RAG Application with LangGraph ReAct Agent

A simple agentic RAG system with LangGraph ReAct agent using OpenAI for HR policies and employee management.

## 1. Install Required Libraries

In [31]:
!pip install -q langgraph langchain langchain-openai langchain-community
!pip install -q chromadb
!pip install -q langchain-chroma
!pip install -q pypdf


In [5]:
# Remove ALL preinstalled OpenTelemetry packages
!pip uninstall -y opentelemetry-sdk opentelemetry-api \
    opentelemetry-semantic-conventions opentelemetry-exporter-otlp \
    opentelemetry-exporter-otlp-proto-grpc opentelemetry-proto \
    opentelemetry-instrumentation opentelemetry-instrumentation-logging \
    opentelemetry-instrumentation-threading opentelemetry-instrumentation-asyncio

# Install Patronus and LangChain instrumentation first
!pip install patronus openinference-instrumentation-langchain langchain-mistralai langgraph

# Pin *all* OTel core packages to the version known to work
!pip install --force-reinstall \
    opentelemetry-api==1.37.0 \
    opentelemetry-sdk==1.37.0 \
    opentelemetry-semantic-conventions==0.58b0 \
    opentelemetry-exporter-otlp-proto-grpc==1.37.0 \
    opentelemetry-exporter-otlp==1.37.0 \
    opentelemetry-proto==1.37.0

# Pin instrumentation packages to compatible versions
!pip install --force-reinstall \
    opentelemetry-instrumentation==0.56b0 \
    opentelemetry-instrumentation-logging==0.56b0 \
    opentelemetry-instrumentation-threading==0.56b0 \
    opentelemetry-instrumentation-asyncio==0.56b0

Found existing installation: opentelemetry-sdk 1.37.0
Uninstalling opentelemetry-sdk-1.37.0:
  Successfully uninstalled opentelemetry-sdk-1.37.0
Found existing installation: opentelemetry-api 1.37.0
Uninstalling opentelemetry-api-1.37.0:
  Successfully uninstalled opentelemetry-api-1.37.0
Found existing installation: opentelemetry-semantic-conventions 0.58b0
Uninstalling opentelemetry-semantic-conventions-0.58b0:
  Successfully uninstalled opentelemetry-semantic-conventions-0.58b0
Found existing installation: opentelemetry-exporter-otlp 1.37.0
Uninstalling opentelemetry-exporter-otlp-1.37.0:
  Successfully uninstalled opentelemetry-exporter-otlp-1.37.0
Found existing installation: opentelemetry-exporter-otlp-proto-grpc 1.37.0
Uninstalling opentelemetry-exporter-otlp-proto-grpc-1.37.0:
  Successfully uninstalled opentelemetry-exporter-otlp-proto-grpc-1.37.0
Found existing installation: opentelemetry-proto 1.37.0
Uninstalling opentelemetry-proto-1.37.0:
  Successfully uninstalled opentel

In [1]:
from openinference.instrumentation.langchain import LangChainInstrumentor
from opentelemetry.instrumentation.threading import ThreadingInstrumentor
from opentelemetry.instrumentation.asyncio import AsyncioInstrumentor

import patronus

patronus.init(
    integrations=[
        LangChainInstrumentor(),
        ThreadingInstrumentor(),
        AsyncioInstrumentor(),
    ]
)

PatronusContext(scope=PatronusScope(service='mani-Inspiron-16-7620-2-in-1', project_name='patronus-projects', app='agentic-rag', experiment_id=None, experiment_name=None), tracer_provider=<opentelemetry.sdk.trace.TracerProvider object at 0x7d81a8add8d0>, logger_provider=<patronus.tracing.logger.LoggerProvider object at 0x7d81a8aa54d0>, api_client_deprecated=<patronus.api.api_client.PatronusAPIClient object at 0x7d81a8a71c50>, api_client=<patronus_api.PatronusAPI object at 0x7d81a90accd0>, async_api_client=<patronus_api.AsyncPatronusAPI object at 0x7d81a8a8bfd0>, exporter=<patronus.evals.exporter.BatchEvaluationExporter object at 0x7d81a8ade5d0>, prompts=PromptsConfig(directory=PosixPath('patronus/prompts'), providers=['local', 'api'], templating_engine='f-string'))

## 2. Import Libraries

In [2]:
import os
import sqlite3
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv


## 3. Configuration

In [3]:
# Set OpenAI API key
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# Initialize OpenAI model
llm = ChatOpenAI(model="gpt-4o", temperature=0)

## 4. Setup SQLite Database for User Records

In [4]:
def initialize_database():
    conn = sqlite3.connect('company_data.db')
    cursor = conn.cursor()
    
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS users (
        user_id INTEGER PRIMARY KEY,
        first_name TEXT,
        last_name TEXT,
        designation TEXT,
        seniority_level TEXT,
        department TEXT,
        hire_date TEXT,
        salary REAL,
        performance_rating TEXT
    )
    ''')
    
    users = [
        (1, 'John', 'Doe', 'Software Engineer', 'Senior', 'Engineering', '2020-01-15', 95000, 'Excellent'),
        (2, 'Jane', 'Smith', 'Product Manager', 'Mid-level', 'Product', '2021-03-20', 85000, 'Good'),
        (3, 'Alice', 'Johnson', 'Data Scientist', 'Junior', 'Data Science', '2023-06-01', 75000, 'Good'),
        (4, 'Bob', 'Williams', 'Senior Architect', 'Principal', 'Engineering', '2018-09-10', 130000, 'Excellent'),
        (5, 'Carol', 'Brown', 'HR Manager', 'Senior', 'Human Resources', '2019-11-05', 90000, 'Excellent')
    ]
    
    cursor.executemany('INSERT OR REPLACE INTO users VALUES (?,?,?,?,?,?,?,?,?)', users)
    conn.commit()
    conn.close()
    print("Database initialized with 5 sample users!")

initialize_database()

Database initialized with 5 sample users!


## 5. Setup Vector Databases for RAG

### 5.1 Initialize Text Splitter

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    length_function=len
)

print("Text splitter initialized!")

Text splitter initialized!


### 5.2 Leave Policy Vector Database

In [6]:
LEAVE_POLICY_PDF_PATH = "/home/mani/PatronusAI/agentic_rag_memory/Company_Leave_Policies_Extended.pdf"

def load_leave_policy_vectorstore(pdf_path):
    if not os.path.exists(pdf_path):
        print(f"Warning: PDF not found at {pdf_path}")
        return None
    
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    split_docs = text_splitter.split_documents(documents)
    
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        collection_name="leave_policies",
        persist_directory="./chroma_db_leave"
    )
    
    print(f"Loaded {len(documents)} pages, split into {len(split_docs)} chunks")
    return vectorstore

leave_vectorstore = load_leave_policy_vectorstore(LEAVE_POLICY_PDF_PATH)

Loaded 7 pages, split into 30 chunks


### 5.3 Promotion and Bonus Policy Vector Database

In [7]:
PROMOTION_BONUS_PDF_PATH = "/home/mani/PatronusAI/agentic_rag_memory/Company_Promotion_Bonus_Policies_Enterprise_Grade.pdf"

def load_promotion_bonus_vectorstore(pdf_path):
    if not os.path.exists(pdf_path):
        print(f"Warning: PDF not found at {pdf_path}")
        return None
    
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    split_docs = text_splitter.split_documents(documents)
    
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        collection_name="promotion_bonus_policies",
        persist_directory="./chroma_db_promotion"
    )
    
    print(f"Loaded {len(documents)} pages, split into {len(split_docs)} chunks")
    return vectorstore

promotion_vectorstore = load_promotion_bonus_vectorstore(PROMOTION_BONUS_PDF_PATH)

Loaded 9 pages, split into 45 chunks


## 6. Define Tools

### 6.1 Tool: Get User Context

In [ ]:
@tool
def get_user_context(user_id: int) -> str:
    """Get user information from database to use as context.
    
    Args:
        user_id: The employee's user ID
        
    Returns:
        User context string with employee information
    """
    conn = sqlite3.connect('company_data.db')
    cursor = conn.cursor()
    cursor.execute('SELECT * FROM users WHERE user_id = ?', (user_id,))
    user = cursor.fetchone()
    conn.close()
    
    if not user:
        return f"User ID {user_id} not found"
    
    return f"""Employee Information\n:Employee: {user[1]} {user[2]}
Designation: {user[3]}
Seniority: {user[4]}
Department: {user[5]}
Performance: {user[8]}
Salary: ${user[7]}"""

### 6.2 Tool: Search Leave Policy

In [ ]:
@tool
def search_leave_policy(question: str, user_context: str) -> str:
    """Search leave policy documents with user context.
    
    Args:
        question: Question about leave policies
        user_context: User information context from get_user_context tool
    """
    if leave_vectorstore is None:
        return "Leave policy PDF not loaded"
    
    # Combine user context with question for better retrieval
    search_query = f"{user_context}\n\n{question}"
    
    retriever = leave_vectorstore.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(search_query)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    return f"Relevant Leave Information Retrieved:\n{context}"

### 6.3 Tool: Search Promotion and Bonus Policy

In [10]:
@tool
def search_promotion_bonus_policy(question: str, user_context: str) -> str:
    """Search promotion and bonus policy documents with user context.
    
    Args:
        question: Question about promotion or bonus
        user_context: User information context from get_user_context tool
    """
    if promotion_vectorstore is None:
        return "Promotion/bonus policy PDF not loaded"
    
    # Combine user context with question for better retrieval
    search_query = f"{user_context}\n\n{question}"
    
    retriever = promotion_vectorstore.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(search_query)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    return f"User Context: {user_context}\n\nRelevant Policy:\n{context}"

## 7. Create ReAct Agent with Memory

In [11]:
tools = [
    get_user_context,
    search_leave_policy,
    search_promotion_bonus_policy
]

system_prompt = """You are an HR assistant agent.

Tool Usage Instructions:
1. FIRST use 'get_user_context' with the user_id to get employee information
2. Use the user context from step 1 when calling RAG tools:
   - 'search_leave_policy': Pass question and user_context
   - 'search_promotion_bonus_policy': Pass question and user_context

Always get user context first before searching policies."""

memory = MemorySaver()

agent = create_react_agent(llm, tools, 
                           prompt=system_prompt, 
                           checkpointer=memory)

print("ReAct Agent created with 3 tools and memory!")

ReAct Agent created with 3 tools and memory!


/tmp/ipykernel_156848/1138622878.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools,


## 8. Query Function

In [12]:
@patronus.traced("agentic-rag-test")
def query_agent(question: str, user_id: int, thread_id: str = "default"):
    """Query the agent
    
    Args:
        question: The question to ask
        user_id: The user ID
        thread_id: Thread ID for conversation history
    """
    # Add user_id to the question context
    full_question = f"User ID: {user_id}\n\n{question}"
    
    config = {"configurable": {"thread_id": thread_id}}
    response = agent.invoke({"messages": [HumanMessage(content=full_question)]}, config)
    
    return response['messages'][-1].content

## 9. Example Usage

### 9.1 Get User Information

In [13]:
result = query_agent("What are my details?", user_id=1)
print(result)

Here are your details:

- **Name:** John Doe
- **Designation:** Software Engineer
- **Seniority:** Senior
- **Department:** Engineering
- **Performance:** Excellent
- **Salary:** $95,000


/home/mani/PatronusAI/agentic_rag_memory/patronus-env/lib/python3.11/site-packages/patronus/tracing/logger.py:193: LogDeprecatedInitWarning: LogRecord init with `trace_id`, `span_id`, and/or `trace_flags` is deprecated since 1.35.0. Use `context` instead.
  record = LogRecord(


### 9.2 Search Leave Policy

In [14]:
result = query_agent("How many days of annual leave do I get?", user_id=1)
print(result)

As a Senior Software Engineer, you are entitled to up to 10 paid days of annual leave per year.


/home/mani/PatronusAI/agentic_rag_memory/patronus-env/lib/python3.11/site-packages/patronus/tracing/logger.py:193: LogDeprecatedInitWarning: LogRecord init with `trace_id`, `span_id`, and/or `trace_flags` is deprecated since 1.35.0. Use `context` instead.
  record = LogRecord(


### 9.3 Search Promotion/Bonus Policy

In [15]:
result = query_agent("Get all my details and tell me if I am eligible for a bonus? How will I get it? How many paternity leaves will I get in a year?", user_id=1)
print(result)

Here are your details and the information you requested:

### Personal Details:
- **Name:** John Doe
- **Designation:** Software Engineer
- **Seniority:** Senior
- **Department:** Engineering
- **Performance:** Excellent
- **Salary:** $95,000

### Bonus Eligibility:
As a Senior Software Engineer, you are eligible for a bonus of up to 25% of your annual base salary. The actual bonus payout is contingent upon company financial performance, your individual performance, and continued employment at the payout date. Bonus payouts can range from 0% to 150% of the target, depending on performance outcomes and company results. Bonuses are prorated for partial-year employment.

### Paternity Leave:
You are entitled to up to 4 weeks of paid paternity leave following the birth or adoption of a child.


/home/mani/PatronusAI/agentic_rag_memory/patronus-env/lib/python3.11/site-packages/patronus/tracing/logger.py:193: LogDeprecatedInitWarning: LogRecord init with `trace_id`, `span_id`, and/or `trace_flags` is deprecated since 1.35.0. Use `context` instead.
  record = LogRecord(


## 10. Test Conversation Memory

In [1]:
# First question
response1 = query_agent("What are my details?", user_id=2, thread_id="conv-1")
print("Question 1:", response1)
print("\n" + "="*80 + "\n")

# Follow-up 1
response2 = query_agent("What leave policies apply to me?", user_id=2, thread_id="conv-1")
print("Question 2 (uses memory):", response2)
print("\n" + "="*80 + "\n")

# Follow-up 2
response3 = query_agent("And bonus?", user_id=2, thread_id="conv-1")
print("Question 3 (uses memory):", response3)

KeyboardInterrupt: 